<a href="https://colab.research.google.com/github/Martazzz/student_performance_project/blob/main/notebooks/01_cleaning_eda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Style and palette**

---
Defined style and palette for the whole project, it defines the colors and parameters for the graphical representations

In [25]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")

plt.rcParams.update(
    {
        "figure.figsize": (9, 5),
        "axes.titlesize": 14,
        "axes.titleweight": "bold",
        "axes.labelsize": 12,
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
    }
)

PALETTE_SCUOLA = {"Public": "#1f77b4", "Private": "#ff7f0e"}
PALETTE_BINARIA = {"Yes": "#2ca02c", "No": "#d62728"}
PALETTE_LIVELLI = ["#9ecae1", "#4292c6", "#084594"]
COLORE_BARRE = "#1f77b4"

# **Reading data**

---
From 20 columns of the dataset, we removed 3 columns: Access_to_Resources, Internet_Access and Parental_Education_Level


In [26]:
raw_data=pd.read_csv("StudentPerformanceFactors.csv", sep=";")

col = ["Hours_Studied", "Attendance", "Parental_Involvement", "Extracurricular_Activities",
        "Sleep_Hours", "Previous_Scores", "Motivation_Level", "Tutoring_Sessions",
        "Family_Income", "Teacher_Quality", "School_Type", "Peer_Influence",
        "Physical_Activity", "Learning_Disabilities", "Distance_from_Home", "Gender", "Exam_Score"]

data = raw_data[col].copy()

print(f"The dataset has {data.shape[0]} rows and {data.shape[1]} columns")

The dataset has 6607 rows and 17 columns


# **Cleaning data**

---
We search for **null values** in the data and substitute it with the mode of the parameter

In [27]:
print("Null values per column:")
print(data.isna().sum()[data.isna().sum() > 0])

teacher_quality_mode = data["Teacher_Quality"].mode()[0]
data["Teacher_Quality"] = data["Teacher_Quality"].fillna(teacher_quality_mode)

distance_mode = data["Distance_from_Home"].mode()[0]
data["Distance_from_Home"] = data["Distance_from_Home"].fillna(distance_mode)

print("\nRemaining missing values in dataset:", data.isna().sum().sum())

Null values per column:
Teacher_Quality       78
Distance_from_Home    67
dtype: int64

Remaining missing values in dataset: 0


---
We search for **anomalies** in the data, such as a score higher than 100 (the exam score is between 0 and 100) and eliminate these values



In [28]:
print(data[["Exam_Score", "Attendance", "Hours_Studied"]].describe())

data = data[(data["Exam_Score"] >= 0) & (data["Exam_Score"] <= 100)]
data = data[(data["Attendance"] >= 0) & (data["Attendance"] <= 100)]


        Exam_Score   Attendance  Hours_Studied
count  6607.000000  6607.000000    6607.000000
mean     67.235659    79.977448      19.975329
std       3.890456    11.547475       5.990594
min      55.000000    60.000000       1.000000
25%      65.000000    70.000000      16.000000
50%      67.000000    80.000000      20.000000
75%      69.000000    90.000000      24.000000
max     101.000000   100.000000      44.000000


---
We now consider the **categorical and binary data**, assigning to each category a numerical representation. In particular we associated:


*   No:0, Yes:1
*   Public:0, Private:1
*   Female:0, Male:1
*   Low:0, Medium:1, High:2
*   Negative:0, Neutral:1, Positive:2
*   Near:0, Moderate:1, Far:2

Then we also calcolated a parameter, ***Score_Change***, that represents the difference between the new score and the previous one.

Another important computation of the data is on the *Hours_Studied*, we decided to bin the hours into 4 distinct quartiles ("Low (Q1)", "Medium-Low (Q2)", "Medium-High (Q3)", "High (Q4)")





In [29]:
data["Extracurricular_Activities_num"] = data["Extracurricular_Activities"].map({"No": 0, "Yes": 1})
data["School_Type_num"] = data["School_Type"].map({"Public": 0, "Private": 1})
data["Learning_Disabilities_num"] = data["Learning_Disabilities"].map({"No": 0, "Yes": 1})
data["Gender_num"] = data["Gender"].map({"Female": 0, "Male": 1})

mapping_levels_3 = {"Low": 0, "Medium": 1, "High": 2}
mapping_peer = {"Negative": 0, "Neutral": 1, "Positive": 2}
mapping_distance = {"Near": 0, "Moderate": 1, "Far": 2}

data["Parental_Involvement_num"] = data["Parental_Involvement"].map(mapping_levels_3)
data["Motivation_Level_num"] = data["Motivation_Level"].map(mapping_levels_3)
data["Family_Income_num"] = data["Family_Income"].map(mapping_levels_3)
data["Teacher_Quality_num"] = data["Teacher_Quality"].map(mapping_levels_3)
data["Peer_Influence_num"] = data["Peer_Influence"].map(mapping_peer)
data["Distance_from_Home_num"] = data["Distance_from_Home"].map(mapping_distance)

data["Score_Change"] = data["Exam_Score"] - data["Previous_Scores"]

data["Hours_Studied_Bins"] = pd.qcut(data["Hours_Studied"], q=4, labels=["Low (Q1)", "Medium-Low (Q2)", "Medium-High (Q3)", "High (Q4)"])

# **Clean dataset**

In [30]:
nome_file = "student_performance_cleaned.csv"
data.to_csv(nome_file, index=False)